# Tugas Praktikum Week 10: CNN dan Transfer Learning dengan YOLOv8
### Klasifikasi Tingkat Kematangan Pisang

| | |
|---|---|
| **Nama** | Tika Altiora Sitanggang |
| **Mata kuliah** | Kecerdasan Buatan |
| **Dosen** | Tegar Arifin Prasetyo, S.Si., M.Si |
| **Materi** | Praktikum CNN dan Transfer Learning (YOLOv8) |
| **Dataset** | Banana Ripeness Classification, Roboflow Universe (versi 4) |

---

## Soal
Menjelaskan secara rinci kode program beserta hasilnya. Di slide teori sudah ada gambaran umumnya, jadi di tugas ini saya menjelaskan maksud dan tujuan dari setiap bagian kode dan juga membaca hasil yang keluar.

## Cara membaca notebook ini
Semua sel kode dan output di bawah ini adalah notebook praktikum asli dari dosen. Kodenya tidak saya ubah. Satu-satunya perubahan adalah API key Roboflow di sel dataset yang saya samarkan, karena itu kunci pribadi akun.

Yang saya tambahkan adalah penjelasan. Setiap sel kode diberi penjelasan dengan pola yang sama supaya mudah diperiksa:

- **Tujuan**: apa yang mau dicapai oleh sel tersebut dan kenapa langkah itu perlu.
- **Penjelasan kode**: arti tiap baris atau tiap parameter.
- **Hasil dan analisis**: apa yang keluar, cara membacanya, dan kesimpulan kecilnya.

## Gambaran umum
Masalah yang dikerjakan adalah **klasifikasi gambar pisang ke dalam 4 kelas kematangan**:

| Kelas | Artinya | Ciri di gambar |
|---|---|---|
| `unripe` | belum matang | kulit hijau |
| `ripe` | matang | kulit kuning, bintik cokelat sedikit |
| `overripe` | terlalu matang | kuning dengan banyak bintik atau bercak cokelat |
| `rotten` | busuk | dominan cokelat gelap atau hitam |

Ada dua pendekatan yang dibandingkan:

1. **Bagian A, CNN buatan sendiri** (TensorFlow/Keras). Arsitekturnya disusun manual dan dilatih dari nol, jadi bobotnya mulai dari angka acak.
2. **Bagian B, YOLOv8 dengan transfer learning** (Ultralytics). Memakai model `yolov8x-cls` yang sudah pernah dilatih pada dataset besar, lalu disesuaikan ke 4 kelas pisang.

## Daftar isi
- **Bagian A**: CNN dari nol
  - A.1 Cek GPU, A.2 Dataset, A.3 Preprocessing, A.4 Arsitektur model, A.5 Training, A.6 Evaluasi, A.7 Kurva training, A.8 Inferensi
- **Bagian B**: YOLOv8
  - B.1 Instalasi, B.2 Inferensi model pretrained, B.3 Custom training, B.4 Validasi, B.5 Inferensi model custom
- **Bagian C**: Perbandingan CNN dan YOLOv8
- **Bagian D**: Kesimpulan
- **Bagian E**: Catatan temuan dan saran perbaikan

---
# Bagian A. CNN yang Dibuat Sendiri (TensorFlow/Keras)

Dari slide teori, CNN (Convolutional Neural Network) berbeda dengan jaringan *fully connected* (MLP) karena CNN bisa menangkap **hubungan spasial antar piksel**. Piksel yang berdekatan dianggap saling berhubungan, jadi tepi, tekstur, dan pola bisa dikenali. Kalau gambar langsung diratakan ke MLP, informasi letak ini hilang dan jumlah parameternya sangat besar.

Komponen utama CNN dari slide, dan semuanya muncul di kode bagian ini:

| Komponen | Fungsi | Muncul di kode sebagai |
|---|---|---|
| Convolutional layer | mengekstraksi fitur (tepi, tekstur, pola) memakai filter/kernel | `Conv2D` |
| Activation function | memberi sifat non-linear | `activation='relu'` |
| Pooling layer | memperkecil dimensi peta fitur | `MaxPooling2D` |
| Fully connected layer | memutuskan kelas dari fitur yang sudah diekstrak | `Flatten`, `Dense` |
| Softmax | mengubah output jadi probabilitas tiap kelas | `Dense(4, activation='softmax')` |

## A.1 Mengecek GPU

**Tujuan.** Memastikan Google Colab sudah memberi akses GPU sebelum training dimulai.

**Penjelasan kode.**
- `!nvidia-smi`: tanda `!` artinya perintah dijalankan di terminal Linux milik Colab, bukan sebagai kode Python. `nvidia-smi` adalah program bawaan NVIDIA yang menampilkan kondisi GPU (jenis, memori, suhu, penggunaan).
- Alasannya: melatih CNN pada ribuan gambar 224x224x3 berisi jutaan operasi perkalian matriks. GPU bisa mengerjakannya secara paralel, jadi jauh lebih cepat daripada CPU. Kalau GPU belum aktif, di slide dijelaskan caranya: **Edit, Pengaturan notebook, Akselerator perangkat keras, pilih T4 GPU, lalu Simpan**.

In [1]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


**Hasil dan analisis.**

| Informasi di output | Nilai | Artinya |
|---|---|---|
| GPU | Tesla T4 | GPU gratis dari Colab, cukup untuk tugas ini |
| Memory-Usage | 0MiB / 15360MiB | VRAM total sekitar 15 GB, belum terpakai sama sekali |
| GPU-Util | 0% | GPU masih menganggur karena belum ada proses |
| Driver / CUDA | 535.104.05 / 12.2 | versi driver dan CUDA yang tersedia |
| Processes | No running processes found | belum ada program yang memakai GPU |

Kesimpulannya GPU sudah aktif dan siap dipakai. Angka 0% dan 0MiB wajar karena pengecekan ini dilakukan sebelum training.

## A.2 Mengunduh Dataset dari Roboflow

**Tujuan.** Mengambil dataset *Banana Ripeness Classification* dari Roboflow Universe langsung ke Colab, lalu merapikan nama folder supaya cocok dengan kode training berikutnya.

**Penjelasan kode.**

| Baris | Fungsinya |
|---|---|
| `!pip install roboflow` | memasang library `roboflow` agar Python bisa mengakses dataset di Roboflow |
| `from roboflow import Roboflow` | memanggil kelas utama dari library itu |
| `rf = Roboflow(api_key=...)` | login memakai API key akun. API key ini seperti password, makanya di notebook ini saya ganti jadi tanda bintang |
| `rf.workspace("roboflow-universe-projects").project("banana-ripeness-classification")` | memilih workspace dan project yang dituju |
| `version = project.version(4)` | memilih versi dataset ke-4 |
| `dataset = version.download("folder")` | mengunduh dan mengekstrak dataset. Format `"folder"` artinya gambar disusun dalam folder per kelas, cocok untuk `flow_from_directory` (Keras) dan mode classify (YOLO) |
| `!mv .../valid .../val` | mengganti nama folder `valid` menjadi `val` |

Variabel `dataset` menyimpan lokasi hasil unduhan di `dataset.location`, yaitu `/content/Banana-Ripeness-Classification-4`. Variabel ini dipakai di hampir semua sel setelahnya.

**Kenapa folder `valid` diganti jadi `val`?** Roboflow menamai folder validasi `valid`, tapi YOLOv8 mode classify mencari folder bernama `val`. Terlihat di log YOLO nanti: `val: ... found 1123 images in 4 classes`. Tanpa penggantian nama ini, YOLO tidak akan menemukan data validasinya.

Struktur folder setelah langkah ini kira-kira:

```
Banana-Ripeness-Classification-4/
├── train/   (11.793 gambar)
│   ├── overripe/  ├── ripe/  ├── rotten/  └── unripe/
├── val/     (1.123 gambar)
│   └── (4 folder kelas yang sama)
└── test/    (562 gambar)
    └── (4 folder kelas yang sama)
```

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="********** (disamarkan)")
project = rf.workspace("roboflow-universe-projects").project("banana-ripeness-classification")
version = project.version(4)
dataset = version.download("folder")

!mv /content/Banana-Ripeness-Classification-4/valid /content/Banana-Ripeness-Classification-4/val

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/7.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.6 MB 1.3 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/7.6 MB 1.8 MB/s eta 0:00:04
   --------- ------------------------------ 1.8/7.6 MB 2.5 MB/s eta 0:00:03
   ------------- -------------------------- 2.6/7.6 MB 2.8 MB/s eta 0:00:02
   --------------- ------------------------ 2.9/7.6 MB 2.6 MB/s eta 0:00:02
   ------------------ --------------------- 3.4/7.6 MB 2.5 MB/s eta 0:00:02
   -------------------- ------------------- 3.9/7.6 MB 2.6 MB/s eta 0:00:02
   ------------------------ --------------- 4.7/7.6 MB 2.6 MB/s eta 0:00:02
   --------------------------- ------------ 5.2/7.6 MB 2.7 MB/s eta 0:00:01
   ----------------------------- ---------- 5.5/7.6 MB 2.6 MB/s eta 0:00:01
   ------------------------------- -------- 6.0/7.6 MB 2.5 MB/s eta 0:00:01
   -----------------------

RuntimeError: {"error":{"message":"This API key does not exist (or has been revoked).","status":401,"type":"OAuthException","hint":"You may retrieve your API key via the Roboflow Dashboard. Go to Account > Roboflow Keys to retrieve yours."}}

**Hasil dan analisis.**
- Log `pip install` kebanyakan berisi *Requirement already satisfied*, artinya library pendukung (matplotlib, numpy, dll.) sudah ada di Colab. Yang benar-benar diunduh hanya `roboflow` dan beberapa paket kecil.
- Progress *Downloading Dataset Version Zip* sampai 100% menandakan file zip dataset selesai diunduh.
- Progress *Extracting ... 13495/13495* berarti ada 13.495 file yang diekstrak. Jumlah gambar sebenarnya 11.793 (train) + 1.123 (val) + 562 (test) = 13.478. Sisa 17 file kemungkinan file keterangan dataset (seperti README), bukan gambar.
- Pembagian datanya sekitar 87% train, 8% validasi, 4% test. Data train paling besar karena model butuh banyak contoh untuk belajar.
- Gambar dataset ini sudah diaugmentasi oleh Roboflow, kelihatan dari beberapa gambar yang berputar dan punya sudut hitam (terlihat di hasil prediksi YOLO di bagian B.5).

## A.3 Preprocessing Data dan Augmentasi

**Tujuan.** Menyiapkan gambar agar bisa dibaca model: ukurannya diseragamkan, nilai pikselnya dinormalisasi, dan data latih diperbanyak variasinya lewat augmentasi. Di slide ada tiga hal yang disebut: *resizing*, *normalization*, dan *augmentation*. Ketiganya ada di sel ini.

**Penjelasan kode.**

`ImageDataGenerator` adalah alat dari Keras yang membaca gambar dari folder, mengolahnya, lalu menyuplai ke model per kelompok kecil (*batch*), sehingga tidak semua 11 ribu gambar dimuat sekaligus ke memori.

`train_dir` dan `valid_dir` adalah alamat folder train dan val, dibuat dari `dataset.location`.

Parameter pada `train_datagen`:

| Parameter | Arti | Kenapa dipakai |
|---|---|---|
| `rescale=1./255` | semua nilai piksel dibagi 255, jadi dari rentang 0-255 menjadi 0-1 | angka kecil membuat training lebih stabil dan cepat (normalisasi di slide) |
| `rotation_range=20` | gambar diputar acak sampai 20 derajat | foto pisang bisa diambil dari sudut berbeda |
| `width_shift_range=0.2` | digeser ke kiri/kanan sampai 20% lebar | posisi pisang di foto tidak selalu di tengah |
| `height_shift_range=0.2` | digeser ke atas/bawah sampai 20% tinggi | alasannya sama |
| `shear_range=0.2` | gambar dimiringkan (perubahan sudut pandang) | meniru perubahan perspektif |
| `zoom_range=0.2` | diperbesar atau diperkecil acak sampai 20% | jarak kamera ke pisang bisa berbeda |
| `horizontal_flip=True` | dicerminkan kiri-kanan | pisang tetap pisang walaupun dibalik |

Augmentasi dilakukan *on the fly*, yaitu saat training berjalan. Tidak ada file baru yang dibuat, dan di setiap epoch gambar yang sama bisa muncul dengan variasi berbeda. Tujuannya seperti yang ada di slide: **meningkatkan generalisasi model dan mengurangi overfitting**.

`valid_datagen` hanya melakukan `rescale`. Data validasi tidak boleh diaugmentasi karena fungsinya untuk menguji model pada data apa adanya. Kalau ikut diacak, hasil ujinya jadi tidak adil dan tidak mewakili kondisi nyata.

`flow_from_directory(...)` yang membuat generatornya:
- `target_size=(224, 224)`: semua gambar diubah ke 224x224 piksel (*resizing*), karena model butuh input berukuran tetap.
- `batch_size=32`: model menerima 32 gambar sekali jalan, lalu bobot diperbarui. Kecil supaya muat di memori dan pembaruan bobotnya cukup sering.
- `class_mode='categorical'`: label dibuat dalam bentuk *one-hot* (misalnya kelas ke-2 menjadi `[0, 0, 1, 0]`). Bentuk ini yang dibutuhkan lapisan softmax dan loss `categorical_crossentropy`.
- Nama kelas diambil otomatis dari nama subfolder dan diurutkan abjad, jadi **overripe = 0, ripe = 1, rotten = 2, unripe = 3**. Urutan ini penting nanti saat membaca classification report.
- Secara bawaan `flow_from_directory` memakai `shuffle=True` (urutan data diacak). Ini sepele, tapi punya dampak di bagian evaluasi (A.6).

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Direktori dataset
train_dir = dataset.location + "/train"
valid_dir = dataset.location + "/val"

# Augmentasi data dan preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Normalisasi gambar
    rotation_range=20,      # Rotasi acak
    width_shift_range=0.2,  # Pergeseran horizontal
    height_shift_range=0.2, # Pergeseran vertikal
    shear_range=0.2,        # Perubahan sudut
    zoom_range=0.2,         # Zoom acak
    horizontal_flip=True    # Membalik gambar secara horizontal
)

# Untuk set validasi, kita hanya melakukan rescale
valid_datagen = ImageDataGenerator(rescale=1./255)

# Membuat data generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),   # Ubah ukuran gambar
    batch_size=32,
    class_mode='categorical'  # Klasifikasi multi-kelas
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


**Hasil dan analisis.**
- `Found 11793 images belonging to 4 classes.` berarti folder train berisi 11.793 gambar dari 4 kelas.
- `Found 1123 images belonging to 4 classes.` berarti folder val berisi 1.123 gambar dari 4 kelas.
- Dari angka ini bisa dihitung jumlah batch: `11793 / 32 = 368,5` jadi 369 batch untuk train, dan `1123 / 32 = 35,1` jadi 36 batch untuk validasi. Angka ini akan dipakai lagi saat membaca hasil training di A.5, karena ada kejanggalan kecil yang berkaitan dengan pembagian batch.
- Data test (562 gambar) tidak dibuat generatornya di bagian CNN. Data ini baru dipakai di bagian YOLO.

## A.4 Membangun Arsitektur CNN

**Tujuan.** Menyusun jaringan CNN yang menerima gambar pisang 224x224x3 dan mengeluarkan probabilitas untuk 4 kelas kematangan.

**Penjelasan kode.**

`Sequential()` berarti layer disusun berurutan, output layer sebelumnya jadi input layer sesudahnya.

Tiga blok konvolusi, tiap blok terdiri dari `Conv2D` lalu `MaxPooling2D`:

| Blok | Kode | Penjelasan |
|---|---|---|
| 1 | `Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3))` | 32 filter berukuran 3x3. Input berupa gambar 224x224 dengan 3 channel warna (RGB). Aktivasi ReLU, `f(x) = max(0, x)`, mengubah nilai negatif jadi nol seperti di slide |
| | `MaxPooling2D(pool_size=(2, 2))` | mengambil nilai terbesar di tiap kotak 2x2, sehingga ukuran peta fitur jadi setengahnya |
| 2 | `Conv2D(64, (3, 3), activation='relu')` + `MaxPooling2D` | 64 filter, mengenali pola yang lebih rumit dari hasil blok 1 |
| 3 | `Conv2D(128, (3, 3), activation='relu')` + `MaxPooling2D` | 128 filter, pola yang lebih rumit lagi |

Jumlah filter sengaja dinaikkan (32, 64, 128) karena layer awal biasanya menangkap fitur sederhana seperti tepi dan warna, lalu layer yang lebih dalam menggabungkannya menjadi pola yang lebih kompleks seperti bercak atau bentuk ujung pisang. Sementara ukuran gambar terus mengecil karena pooling.

Bagian akhir (klasifikasi):

| Kode | Penjelasan |
|---|---|
| `Flatten()` | meratakan peta fitur 3 dimensi menjadi satu vektor panjang, supaya bisa masuk ke layer fully connected |
| `Dense(128, activation='relu')` | fully connected layer berisi 128 neuron |
| `Dropout(0.5)` | saat training, 50% neuron dimatikan secara acak di setiap langkah. Model jadi tidak bergantung pada neuron tertentu, sehingga lebih tahan overfitting (dijelaskan di slide) |
| `Dense(4, activation='softmax')` | 4 neuron output, satu per kelas. Softmax mengubah angka mentah jadi probabilitas yang totalnya 1 |

`model.compile(...)`:
- `optimizer='adam'`: algoritma yang memperbarui bobot. Konsepnya sama dengan SGD di slide backpropagation (`w_baru = w_lama - learning rate x gradien`), tetapi Adam menyesuaikan learning rate untuk tiap bobot secara otomatis sehingga biasanya lebih cepat konvergen.
- `loss='categorical_crossentropy'`: fungsi loss untuk klasifikasi multi-kelas (cross-entropy di slide). Pasangan dari label one-hot dan softmax.
- `metrics=['accuracy']`: ukuran yang ditampilkan saat training, yaitu persentase tebakan benar.

`model.summary()` menampilkan ringkasan tiap layer, bentuk outputnya, dan jumlah parameternya.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Membangun model CNN
model = Sequential()

# Lapisan Convolutional pertama
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lapisan Convolutional kedua
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lapisan Convolutional ketiga
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lapisan Flatten
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))  # Dropout untuk mencegah overfitting

# Lapisan Output
model.add(Dense(4, activation='softmax'))  # 4 kelas untuk kematangan pisang

# Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Menampilkan ringkasan model
model.summary()

**Hasil dan analisis.**

*Pesan "Do not pass an `input_shape` ..."* hanya peringatan dari Keras versi baru yang menyarankan memakai `Input(shape)` sebagai layer pertama. Model tetap berjalan normal.

Output summary bisa dicek dengan perhitungan manual. Konvolusi di sini memakai `padding='valid'` (bawaan, tanpa padding) dan stride 1, jadi ukuran gambar berkurang 2 di setiap konvolusi.

| Layer | Perhitungan ukuran | Output | Perhitungan parameter | Param |
|---|---|---|---|---|
| conv2d | 224 - 3 + 1 = 222 | (222, 222, 32) | (3x3x3 + 1) x 32 | 896 |
| max_pooling2d | 222 / 2 = 111 | (111, 111, 32) | tidak ada bobot | 0 |
| conv2d_1 | 111 - 3 + 1 = 109 | (109, 109, 64) | (3x3x32 + 1) x 64 | 18.496 |
| max_pooling2d_1 | 109 / 2 = 54 (dibulatkan ke bawah) | (54, 54, 64) | tidak ada bobot | 0 |
| conv2d_2 | 54 - 3 + 1 = 52 | (52, 52, 128) | (3x3x64 + 1) x 128 | 73.856 |
| max_pooling2d_2 | 52 / 2 = 26 | (26, 26, 128) | tidak ada bobot | 0 |
| flatten | 26 x 26 x 128 | (86.528) | tidak ada bobot | 0 |
| dense | 128 neuron | (128) | (86.528 + 1) x 128 | 11.075.712 |
| dropout | sama dengan input | (128) | tidak ada bobot | 0 |
| dense_1 | 4 neuron | (4) | (128 + 1) x 4 | 516 |

Rumus parameter konvolusi: `(tinggi filter x lebar filter x channel input + 1 bias) x jumlah filter`. Rumus parameter Dense: `(jumlah input + 1 bias) x jumlah neuron`.

Total parameter = 896 + 18.496 + 73.856 + 11.075.712 + 516 = **11.169.476** (sekitar 42,61 MB kalau tiap parameter disimpan sebagai angka 4 byte). Hasilnya sama dengan yang tertera di output.

Hal yang menarik dari tabel ini:
1. Hampir seluruh parameter (11.075.712 dari 11.169.476, sekitar 99,2%) ada di satu layer saja, yaitu `Dense(128)` setelah `Flatten`. Tiga blok konvolusi hanya menyumbang sekitar 93 ribu parameter. Ini menunjukkan keuntungan konvolusi: filter 3x3 dipakai berulang di seluruh gambar (*weight sharing*), sehingga parameternya sedikit.
2. Karena layer Dense sebesar itu, risiko overfitting tinggi. Itu alasan `Dropout(0.5)` dipasang tepat setelahnya.
3. `Non-trainable params: 0` karena semua bobot dilatih dari awal. Nanti bandingkan dengan YOLO yang sebagian layernya dibekukan.

## A.5 Melatih Model CNN

**Tujuan.** Melatih CNN dengan data train, sambil memantau performa di data validasi pada setiap epoch.

**Penjelasan kode.**

`history = model.fit(...)` menjalankan proses training dan menyimpan catatan loss dan akurasi tiap epoch ke variabel `history`.

| Parameter | Nilai | Arti |
|---|---|---|
| `train_generator` | | sumber data latih (sudah mencakup augmentasi) |
| `steps_per_epoch` | `11793 // 32 = 368` | jumlah batch yang diproses dalam satu epoch. `//` adalah pembagian bulat |
| `validation_data` | `valid_generator` | data untuk menguji model di akhir setiap epoch |
| `validation_steps` | `1123 // 32 = 35` | jumlah batch validasi per epoch |
| `epochs` | 9 | jumlah putaran training. Satu epoch artinya model sudah melihat seluruh data latih satu kali |

Di setiap langkah (*step*), yang terjadi sama dengan konsep di slide MLP:
1. **Forward propagation**: satu batch (32 gambar) dimasukkan, dihitung sampai menghasilkan prediksi.
2. Prediksi dibandingkan dengan label asli untuk menghitung **loss** (cross-entropy).
3. **Backpropagation**: gradien loss dihitung mundur dari output ke input dengan aturan rantai.
4. **Update bobot** oleh Adam. Lalu proses diulang untuk batch berikutnya.

In [ ]:
# Melatih model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // valid_generator.batch_size,
    epochs=9  # Jumlah epoch
)


**Hasil dan analisis.**

Ringkasan angka dari log di atas:

| Epoch | Waktu | accuracy | loss | val_accuracy | val_loss |
|---|---|---|---|---|---|
| 1 | 178 s | 0,5253 | 1,1277 | 0,8607 | 0,3915 |
| 2 | 1 s | 0,8438 | 0,4817 | 1,0000 | 0,1351 |
| 3 | 148 s | 0,7904 | 0,5188 | 0,8759 | 0,3422 |
| 4 | 0 s | 0,8750 | 0,2662 | 1,0000 | 0,0711 |
| 5 | 149 s | 0,8494 | 0,4136 | 0,9000 | 0,2850 |
| 6 | 0 s | 0,9062 | 0,2739 | 0,6667 | 0,7345 |
| 7 | 201 s | 0,8609 | 0,3788 | 0,8964 | 0,3213 |
| 8 | 0 s | 1,0000 | 0,1333 | 1,0000 | 0,0768 |
| 9 | 147 s | 0,8927 | 0,3069 | 0,8848 | 0,4519 |

**1. Model benar-benar belajar.** Di epoch 1 akurasi training baru 52,5% dengan loss 1,13. Pada epoch 9 akurasi naik ke sekitar 89% dan loss turun ke sekitar 0,31. Akurasi validasi di epoch ganjil (1, 3, 5, 7, 9) stabil di kisaran 86% sampai 90%.

**2. Kenapa akurasi validasi di epoch 1 lebih tinggi dari akurasi training?** Ini kelihatan aneh, tapi ada penjelasannya. Akurasi training dihitung sebagai rata-rata selama epoch, termasuk saat model masih belum pintar di awal. Selain itu, saat training augmentasi dan dropout aktif sehingga soalnya lebih sulit, sedangkan saat validasi data tidak diubah dan dropout dimatikan. Jadi wajar kalau nilai validasi sementara lebih bagus.

**3. Kenapa epoch genap hanya 0 sampai 1 detik?** Epoch 1, 3, 5, 7, 9 memakan 147 sampai 201 detik (sekitar 400 ms per langkah), sedangkan epoch 2, 4, 6, 8 selesai dalam hitungan detik saja. Di log juga ada pesan *"Your input ran out of data; interrupting training"*. Dugaan saya penyebabnya adalah pembagian batch yang tidak pas. Datanya 11.793 gambar sehingga generator punya 369 batch, tetapi `steps_per_epoch` diset 368. Setelah satu epoch penuh tersisa 1 batch, dan di epoch berikutnya generator kehabisan data setelah 1 langkah, sehingga epoch itu terpotong. Generator lalu mulai dari awal lagi, jadi epoch berikutnya normal lagi dan pola ini berulang selang-seling.

Untuk validasi pola yang sama kemungkinan terjadi. Data validasi 1.123 gambar dengan `validation_steps = 35` menyisakan 1123 - 35 x 32 = 3 gambar. Kalau yang dievaluasi di epoch genap cuma 3 gambar, akurasinya hanya bisa 0, 0,33, 0,67, atau 1,00. Nilai yang keluar memang 1,0000 dan 0,6667, cocok dengan dugaan ini.

Akibatnya, **epoch genap tidak bisa dijadikan patokan**. Yang mencerminkan kemampuan model adalah epoch ganjil. (Saran perbaikannya ada di Bagian E.)

**4. Peringatan `PyDataset ... super().__init__`** hanya peringatan teknis versi Keras baru dan tidak memengaruhi hasil.

**5. Waktu.** Satu epoch penuh sekitar 2,5 menit di T4. Total training kira-kira 14 menit.

## A.6 Evaluasi Model: Akurasi, Precision, Recall, F1-Score, dan Confusion Matrix

**Tujuan.** Menilai kualitas model di data validasi, bukan hanya dari satu angka akurasi, tetapi juga performanya di tiap kelas. Ini sesuai bagian *Matriks Evaluasi untuk Klasifikasi* di slide.

**Penjelasan kode.**

*Bagian 1, evaluasi umum*
- `model.evaluate(valid_generator)` menghitung loss dan akurasi model pada seluruh data validasi, lalu hasilnya dicetak dengan `print`.

*Bagian 2, prediksi per gambar*
- `y_pred = model.predict(valid_generator)` menghasilkan array berukuran (1123, 4): untuk tiap gambar, probabilitas keempat kelas.
- `np.argmax(y_pred, axis=1)` mengambil indeks probabilitas terbesar di tiap baris, itulah kelas yang diprediksi (0 sampai 3).
- `y_true = valid_generator.classes` adalah label asli tiap gambar.

*Bagian 3, metrik*
- `classification_report(y_true, y_pred_classes)` menampilkan tabel precision, recall, f1-score, dan support per kelas.
- `confusion_matrix(y_true, y_pred_classes)` membuat matriks 4x4: baris menunjukkan kelas asli dan kolom menunjukkan kelas prediksi.

Rumus yang dipakai (dari slide):

| Metrik | Rumus | Pertanyaan yang dijawab |
|---|---|---|
| Akurasi | `(TP + TN) / (TP + TN + FP + FN)` | dari semua gambar, berapa persen yang ditebak benar? |
| Precision | `TP / (TP + FP)` | kalau model bilang kelas X, seberapa sering benar? |
| Recall | `TP / (TP + FN)` | dari semua gambar kelas X yang asli, berapa persen yang berhasil ditemukan? |
| F1-score | `2 x (Precision x Recall) / (Precision + Recall)` | rata-rata harmonis precision dan recall |

*Bagian 4, menggambar confusion matrix.* Baris-baris `plt...` membuat gambar heatmap:
- `plt.imshow(cm, cmap=plt.cm.Blues)`: menampilkan matriks sebagai gambar, makin besar angkanya makin biru tua.
- `plt.colorbar()`: skala warna di sisi kanan.
- `tick_marks`, `plt.xticks`, `plt.yticks`: menaruh nama kelas (overripe, ripe, rotten, unripe) di sumbu.
- `thresh = cm.max() / 2.`: batas untuk menentukan warna tulisan angka.
- Dua `for` bersarang menulis angka di setiap sel. Tulisan dibuat putih jika selnya gelap (nilai di atas `thresh`) dan hitam jika selnya terang, supaya selalu terbaca.
- `plt.ylabel('True label')` dan `plt.xlabel('Predicted label')`: baris adalah label asli, kolom adalah label prediksi.

In [ ]:
# Model evaluation using precision, recall, confution matrix and f1 score, display confusion matrix in plt

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Evaluasi model pada keseluraun class data validasi
val_loss, val_accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

# Evaluasi model pada setiap class data validasi
y_pred = model.predict(valid_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = valid_generator.classes

# Menghitung metrik evaluasi
report = classification_report(y_true, y_pred_classes)
print(report)

# Membuat confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Menampilkan confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(valid_generator.class_indices))
plt.xticks(tick_marks, valid_generator.class_indices, rotation=45)
plt.yticks(tick_marks, valid_generator.class_indices)

# Menambahkan label pada setiap sel
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()


**Hasil dan analisis.**

**a. Hasil `evaluate`**: Validation Loss = 0,4507 dan Validation Accuracy = 0,8851 (sekitar 88,5%). Angka ini konsisten dengan epoch terakhir training (val_accuracy 0,8848), jadi bisa dipercaya sebagai performa CNN ini.

**b. Hasil `classification_report`**: di sini ada hal yang janggal.

| Kelas | Nama | precision | recall | f1-score | support |
|---|---|---|---|---|---|
| 0 | overripe | 0,18 | 0,14 | 0,16 | 229 |
| 1 | ripe | 0,35 | 0,32 | 0,33 | 339 |
| 2 | rotten | 0,37 | 0,45 | 0,41 | 388 |
| 3 | unripe | 0,11 | 0,11 | 0,11 | 167 |
| | accuracy | | | **0,30** | 1123 |
| | macro avg | 0,25 | 0,25 | 0,25 | 1123 |
| | weighted avg | 0,29 | 0,30 | 0,29 | 1123 |

*Support* adalah jumlah gambar asli per kelas (229 + 339 + 388 + 167 = 1123). *Macro avg* adalah rata-rata biasa dari 4 kelas, sedangkan *weighted avg* adalah rata-rata yang dibobot dengan support.

Akurasi di report hanya **0,30**, padahal `evaluate` tadi menghasilkan **0,885**. Keduanya dihitung dari model dan data validasi yang sama, jadi seharusnya hampir sama. Nilai sekitar 0,25 juga setara dengan menebak asal dari 4 kelas (1 dari 4 = 25%).

**c. Hasil confusion matrix.** Angka di diagonal (sel yang benar) adalah 32, 108, 175, dan 18, jumlahnya 333. Kalau dibagi 1123 hasilnya 0,2965 atau sekitar 0,30, persis sama dengan akurasi di report. Jadi report dan matriksnya konsisten satu sama lain, tapi keduanya tidak cocok dengan hasil `evaluate`.

**d. Penyebab kejanggalan.** Saya menduga penyebabnya adalah pengacakan urutan data. Seperti dijelaskan di A.3, `flow_from_directory` bawaannya memakai `shuffle=True`. Akibatnya urutan gambar yang keluar dari `model.predict(valid_generator)` tidak sama dengan urutan di `valid_generator.classes`. Prediksi gambar ke-1 dibandingkan dengan label gambar lain, sehingga banyak pasangan yang tidak cocok dan hasilnya mendekati tebakan acak.

Bukti pendukung dari confusion matrix: persebaran prediksi di tiap baris mirip satu sama lain (sebagian besar prediksi jatuh ke kolom rotten dan ripe, apa pun kelas aslinya). Artinya prediksi tampak tidak berhubungan dengan label asli, ciri khas pasangan yang tidak sinkron.

**Kesimpulannya**, angka 0,30 di report dan confusion matrix ini bukan kualitas model yang sebenarnya, melainkan akibat urutan data yang tidak sinkron. Kualitas CNN yang sebenarnya lebih dekat ke 0,885 dari `evaluate`. Cara memperbaikinya ada di Bagian E: cukup menambahkan `shuffle=False` pada `valid_generator`. (Slide teori juga menampilkan akurasi sekitar 0,25 untuk bagian ini, jadi kejadiannya sama.)

**e. Cara membaca confusion matrix** (bila urutan sudah benar): baris adalah kelas asli, kolom adalah kelas prediksi. Sel diagonal adalah prediksi yang benar, sel di luar diagonal adalah kesalahan. Misalnya pada matriks di atas, baris `ripe` kolom `rotten` bernilai 135, artinya ada 135 gambar yang aslinya `ripe` dan ditebak `rotten`. Model yang bagus punya angka besar di diagonal dan angka kecil di tempat lain.

## A.7 Visualisasi Kurva Akurasi dan Loss

**Tujuan.** Melihat perkembangan model sepanjang training dalam bentuk grafik, supaya bisa dinilai apakah ada tanda overfitting atau underfitting (seperti di slide).

**Penjelasan kode.**
- `history.history` adalah dictionary berisi daftar nilai per epoch, dengan kunci `'accuracy'`, `'val_accuracy'`, `'loss'`, dan `'val_loss'`. Tiap daftar berisi 9 angka karena ada 9 epoch.
- `plt.plot(...)` menggambar garis, `label=` memberi nama garis, `plt.title` memberi judul, `plt.legend()` menampilkan keterangan garis, `plt.show()` menampilkannya.
- Dibuat dua grafik: satu untuk akurasi dan satu untuk loss. Sumbu x adalah nomor epoch (0 sampai 8, sama dengan epoch 1 sampai 9), sumbu y adalah nilainya.
- Cara membacanya: garis *Training* (biru) menunjukkan performa di data latih, garis *Validation* (oranye) di data validasi. Kalau akurasi training terus naik tetapi validasi turun, itu tanda overfitting. Kalau keduanya rendah, itu tanda underfitting.

In [ ]:
import matplotlib.pyplot as plt

# Visualisasi akurasi
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.show()

# Visualisasi loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()


**Hasil dan analisis.**

*Grafik akurasi:*
- Garis training secara umum naik dari kisaran 0,5-0,6 di epoch awal ke sekitar 0,9 di akhir, walaupun tidak mulus.
- Garis validasi naik-turun tajam: puncak 1,0 di sumbu 1, 3, dan 7, lalu jatuh ke sekitar 0,67 di sumbu 5. Sumbu 1, 3, 5, 7 adalah epoch 2, 4, 6, 8, yaitu epoch genap yang bermasalah di A.5 karena hanya sempat memproses sedikit data. Jadi lonjakan dan jurang itu **bukan** berarti model tiba-tiba sangat bagus atau sangat buruk, tapi akibat evaluasi pada sangat sedikit data.
- Titik yang bisa dipercaya adalah epoch ganjil (sumbu 0, 2, 4, 6, 8), yang validasinya stabil di sekitar 0,86 sampai 0,90.

*Grafik loss:*
- Loss training turun dari sekitar 0,85 ke sekitar 0,3, tren yang bagus. Angka awal di grafik (sekitar 0,85) sedikit berbeda dengan angka epoch 1 di log A.5 (1,13). Beberapa epoch lain juga sedikit berbeda, kemungkinan karena sel training sempat dijalankan lebih dari sekali. Polanya sama, jadi grafik cukup dibaca untuk melihat arah tren saja.
- Loss validasi ikut naik-turun karena alasan yang sama. Di epoch terakhir loss validasi (0,45) lebih tinggi dari loss training (0,31). Selisih kecil seperti ini bisa menjadi tanda awal overfitting, tapi satu titik saja belum cukup untuk menyimpulkannya.

*Kesimpulan.* Akurasi training dan validasi di epoch ganjil berdekatan (sekitar 0,89), jadi tidak ada overfitting yang parah. Model juga tidak underfitting karena akurasinya sudah mendekati 90%. Hanya saja grafiknya belum rapi karena masalah batch di atas, dan 9 epoch belum cukup untuk melihat model benar-benar konvergen.

## A.8 Inferensi: Memprediksi Satu Gambar Baru

**Tujuan.** Memakai model yang sudah dilatih untuk menebak kematangan sebuah foto pisang, sebagai contoh penggunaan model di dunia nyata. Langkahnya sama dengan di slide: muat gambar, proses agar sesuai input model, prediksi, lalu tampilkan hasilnya sebagai kelas.

**Penjelasan kode (fungsi `predict_image`).**

| Baris | Fungsinya |
|---|---|
| `image.load_img(image_path, target_size=(224, 224))` | membaca file gambar dan mengubah ukurannya ke 224x224, sama dengan ukuran saat training |
| `image.img_to_array(img)` | mengubah gambar jadi array angka berukuran (224, 224, 3) |
| `np.expand_dims(img_array, axis=0)` | menambah dimensi di depan, jadi (1, 224, 224, 3). Model selalu mengharapkan input berbentuk *batch*, walaupun isinya hanya 1 gambar |
| `img_array /= 255.` | normalisasi piksel ke 0-1. Ini wajib sama dengan preprocessing saat training (`rescale=1./255`). Kalau lupa, nilai input jauh berbeda dan prediksi jadi ngawur |
| `model.predict(img_array)` | menghasilkan 4 probabilitas |
| `np.argmax(prediction)` | mengambil indeks probabilitas tertinggi |
| `list(train_generator.class_indices.keys())` | mengambil daftar nama kelas: overripe, ripe, rotten, unripe. Dari indeks 0 sampai 3 diubah jadi nama |
| `plt.imshow(img)`, `plt.title(...)`, `plt.axis('off')`, `plt.show()` | menampilkan gambar asli dengan judul berisi label prediksi, tanpa sumbu |

Gambar yang dicoba berada di folder `test/ripe/`, jadi label aslinya diketahui: `ripe`.

In [ ]:
# inference using model, and add text to displayed predict image

from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np

def predict_image(image_path):
  img = image.load_img(image_path, target_size=(224, 224))
  img_array = image.img_to_array(img)
  img_array = np.expand_dims(img_array, axis=0)
  img_array /= 255.

  prediction = model.predict(img_array)
  predicted_class = np.argmax(prediction)

  # Get class labels from the generator
  class_labels = list(train_generator.class_indices.keys())

  # Display the image with prediction
  plt.imshow(img)
  plt.title(f"Predicted label: {class_labels[predicted_class]}")
  plt.axis('off')
  plt.show()

# Example usage
image_path = "/content/Banana-Ripeness-Classification-4/test/ripe/musa-acuminata-banana-ad772802-394a-11ec-996c-d8c4975e38aa_jpg.rf.ac1e17757ffe79f22868f68252821f9d.jpg"
predict_image(image_path)


**Hasil dan analisis.**
- `1/1 ... 802ms/step` berarti satu gambar diproses dalam sekitar 0,8 detik. Waktu ini lebih lama dari biasanya karena panggilan pertama termasuk persiapan awal.
- Gambar yang tampil adalah satu pisang kuning dengan beberapa bintik cokelat kecil di latar gelap, dan judulnya `Predicted label: ripe`.
- Prediksi **sesuai** dengan label asli (folder `ripe`), jadi untuk gambar ini model benar.
- Catatan: ini hanya satu contoh dan fungsinya untuk demonstrasi cara memakai model. Untuk menilai kualitas model yang sebenarnya tetap perlu dilihat dari metrik di data banyak (A.6).
- Fungsi ini hanya menampilkan kelas tertinggi, bukan probabilitasnya. Kalau mau tahu seberapa yakin model, nilai `prediction` bisa dicetak juga.

---
# Bagian B. Transfer Learning dengan YOLOv8

## Pengantar
**YOLO** (*You Only Look Once*) adalah keluarga model yang terkenal untuk deteksi objek *real-time*. **YOLOv8** dari Ultralytics dirancang agar cepat, akurat, dan mudah dipakai, dan bisa dipakai untuk deteksi objek, segmentasi, deteksi pose, deteksi objek berorientasi, dan **klasifikasi gambar** (sesuai slide).

Pada praktikum ini YOLOv8 dipakai untuk **klasifikasi**, bukan untuk menggambar kotak deteksi. Itu terlihat dari nama modelnya, `yolov8x-cls`: `cls` berarti *classification*, dan `x` adalah ukuran paling besar di keluarga YOLOv8 (n, s, m, l, x).

## Ingat kembali: Transfer Learning
Di slide, transfer learning adalah teknik memakai **model yang sudah dilatih sebelumnya (pretrained)** untuk tugas baru, terutama saat datanya terbatas. Ada dua cara:
- **Fine-tuning**: menyesuaikan sebagian besar atau seluruh layer model pretrained dengan dataset baru.
- **Freezing**: membekukan beberapa layer awal dan hanya melatih ulang layer akhir, untuk menghemat waktu dan sering meningkatkan akurasi.

Keuntungannya: hemat waktu, memanfaatkan pengetahuan yang sudah dipelajari (fitur umum seperti tepi, tekstur, dan pola), serta mengurangi kebutuhan data yang besar. Di Bagian A, CNN dilatih dari nol. Di Bagian B ini, model sudah punya bekal dari latihan sebelumnya, tinggal diajari membedakan 4 tingkat kematangan pisang.

## Dua cara menjalankan YOLOv8
Seperti di slide, ada antarmuka baris perintah (CLI) dan Python SDK. Notebook ini memakai **CLI** dengan format:

```
yolo TASK MODE ARGS
```
- `TASK` (opsional): jenis tugas, salah satunya `classify`.
- `MODE` (wajib): `train`, `val`, `predict`, `export`, `track`, atau `benchmark`.
- `ARGS` (opsional): pasangan `arg=value` untuk mengubah pengaturan bawaan, contohnya `epochs=5`.

CLI cocok kalau hanya ingin melatih atau memakai model tanpa mengubah kode.

## B.1 Install YOLOv8

**Tujuan.** Memasang library `ultralytics` (pembungkus YOLOv8) dan memastikan lingkungannya siap.

**Penjelasan kode.**
- `!pip install ultralytics`: memasang library lewat pip.
- `from IPython import display` dan `display.clear_output()`: menghapus output instalasi yang sangat panjang supaya notebook tetap rapi.
- `import ultralytics` dan `ultralytics.checks()`: mengecek versi library, Python, PyTorch, GPU, serta kapasitas CPU, RAM, dan disk.

In [ ]:
# Pip install method (recommended)

!pip install ultralytics

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

**Hasil dan analisis.**

| Informasi | Nilai | Artinya |
|---|---|---|
| Ultralytics | YOLOv8.2.99 | versi library yang terpasang |
| Python | 3.10.12 | versi Python di Colab |
| torch | 2.4.1+cu121 | PyTorch dengan dukungan CUDA 12.1. YOLOv8 dibangun di atas **PyTorch**, berbeda dengan Bagian A yang memakai TensorFlow/Keras |
| CUDA:0 | Tesla T4, 15102MiB | GPU terdeteksi dan siap dipakai |
| Sumber daya | 2 CPU, 12,7 GB RAM, disk 32,8 / 112,6 GB | kapasitas mesin Colab |

Tulisan *Setup complete* menandakan semuanya siap. (Versi di slide 8.2.92, di sini 8.2.99, hanya beda versi kecil.)

## B.2 Inferensi dengan Model Pretrained

**Tujuan.** Mencoba dulu model `yolov8x-cls` bawaan tanpa dilatih ulang. Ini untuk melihat kemampuan awal model pretrained dan memastikan YOLO bisa berjalan.

**Penjelasan kode.**

`!yolo task=classify mode=predict model=yolov8x-cls.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg'`

| Bagian | Arti |
|---|---|
| `task=classify` | tugasnya klasifikasi gambar |
| `mode=predict` | mode inferensi (menebak gambar) |
| `model=yolov8x-cls.pt` | file bobot model pretrained. Jika belum ada, diunduh otomatis |
| `conf=0.25` | batas keyakinan (*confidence threshold*). Untuk klasifikasi pengaruhnya kecil karena yang ditampilkan adalah 5 kelas teratas |
| `source=...` | gambar yang diprediksi, di sini berupa URL foto anjing |

Hasil prediksi disimpan di folder `runs/classify/predict`. Sel berikutnya (`Image(filename=...)`) menampilkan gambar hasilnya di notebook, dengan tinggi 600 piksel.

In [ ]:
!yolo task=classify mode=predict model=yolov8x-cls.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg'

**Hasil dan analisis (log di atas).**
- `Downloading ... yolov8x-cls.pt` (110 MB): bobot model diunduh otomatis.
- `YOLOv8x-cls summary (fused): 133 layers, 57.404.200 parameters, 154.8 GFLOPs`: modelnya besar, sekitar 57 juta parameter (bandingkan dengan CNN buatan sendiri yang 11 juta). Kata *fused* artinya layer konvolusi dan batch normalization sudah digabung agar inferensi lebih cepat. GFLOPs adalah ukuran beban hitungnya.
- Hasil tebakan: **beagle 0,69, Walker_hound 0,12, basset 0,10, English_foxhound 0,06, bloodhound 0,01**. Model paling yakin ini adalah beagle (69%), dan empat kelas sisanya juga jenis anjing yang mirip. Hasil ini masuk akal karena memang foto anjing beagle.
- Waktu: preprocess 170,6 ms, inferensi hanya 32,1 ms per gambar, jadi sangat cepat.
- Daftar kelas yang keluar adalah **nama ras anjing**, dan model ini punya 1000 kelas (di log training nanti ada tulisan `nc=1000`). Ini menunjukkan model `-cls` dilatih pada dataset klasifikasi besar dengan 1000 kelas (ImageNet), bukan dataset COCO. Judul "COCO" di notebook kemungkinan ikut template bawaan.
- Poinnya: model ini **belum tahu** apa itu pisang matang atau busuk, karena kelas yang dia kenal bukan itu. Karenanya perlu *custom training* di B.3.

In [ ]:
from IPython.display import Image
Image(filename='runs/classify/predict/dog.jpeg', height=600)


Gambar di atas adalah hasil prediksi yang disimpan YOLO: foto seekor anjing beagle yang sedang digendong di punggung seseorang, dengan lima label teratas beserta nilai probabilitasnya ditulis di pojok kiri atas gambar. Isinya sama persis dengan log di atas.

## B.3 Custom Training (Transfer Learning pada Dataset Pisang)

**Tujuan.** Melatih ulang `yolov8x-cls` agar mengenali 4 kelas kematangan pisang. Ini inti praktikum transfer learning: bobot pretrained dipakai sebagai titik awal, bukan mulai dari nol.

**Penjelasan kode.**

`!yolo task=classify mode=train model=yolov8x-cls.pt data={dataset.location} epochs=5 imgsz=224 batch=32 freeze=5`

| Argumen | Arti |
|---|---|
| `mode=train` | mode pelatihan |
| `model=yolov8x-cls.pt` | titik awal berupa model pretrained |
| `data={dataset.location}` | lokasi dataset. Tanda kurung kurawal membuat Colab mengganti dengan isi variabel Python `dataset.location`. Mode classify mengharapkan folder `train`, `val` (dan `test`) berisi folder per kelas, dan itu sudah disiapkan di A.2 |
| `epochs=5` | 5 putaran training |
| `imgsz=224` | ukuran gambar 224x224, sama seperti Bagian A |
| `batch=32` | 32 gambar per langkah, sama seperti Bagian A |
| `freeze=5` | **membekukan 5 blok layer pertama**. Ini penerapan *freezing* dari slide |

**Tentang `freeze=5`.** Layer awal pada model pretrained sudah belajar fitur umum seperti garis, tepi, dan tekstur, yang berguna untuk gambar apa pun. Layer-layer itu dibekukan (bobotnya tidak diubah, setara learning rate = 0 seperti pada gambar ilustrasi di slide), sedangkan layer di belakangnya tetap dilatih (learning rate lebih dari 0) supaya menyesuaikan diri dengan pisang. Hasilnya training lebih cepat dan lebih stabil, serta tidak merusak pengetahuan dasar yang sudah bagus.

In [ ]:
!yolo task=classify mode=train model=yolov8x-cls.pt data={dataset.location} epochs=5 imgsz=224 batch=32 freeze=5

**Hasil dan analisis (membaca log training).**

**1. Persiapan model.**
- `Overriding model.yaml nc=1000 with nc=4`: jumlah kelas output diganti dari 1000 menjadi 4. Layer klasifikasi terakhir dibuat baru untuk 4 kelas pisang.
- `Transferred 300/302 items from pretrained weights`: dari 302 kelompok bobot, 300 berhasil disalin dari model pretrained. Dua sisanya kemungkinan bobot dan bias layer terakhir, yang tidak bisa disalin karena ukurannya berbeda (1000 kelas menjadi 4 kelas). Ini yang justru kita inginkan: bagian dalam memakai pengalaman lama, bagian output dilatih baru.
- Baris `Freezing layer 'model.0...'` sampai `model.4...` muncul 75 kali. Ini daftar bobot (konvolusi dan batch norm) milik blok 0 sampai 4 yang dibekukan, sesuai `freeze=5`.
- `AMP: checks passed`: AMP (*Automatic Mixed Precision*) memakai angka berpresisi lebih rendah (float16) di bagian tertentu agar training lebih cepat dan hemat memori. Sebelum dipakai, Ultralytics mengujinya dengan model kecil `yolov8n` (karena itu ada unduhan 6,25 MB).
- Pemindaian data: `11793 images` train dan `1123 images` val, tidak ada file rusak (`0 corrupt`).
- `optimizer: AdamW(lr=0.000714, momentum=0.9)`: karena `optimizer=auto`, Ultralytics memilih sendiri optimizer dan learning rate-nya. AdamW adalah varian Adam dengan perlakuan *weight decay* yang lebih baik. Learning rate yang kecil (sekitar 0,0007) cocok untuk transfer learning agar bobot pretrained tidak berubah drastis.

**2. Perkembangan per epoch.**

| Epoch | Loss (train) | top1_acc (val) | Waktu |
|---|---|---|---|
| 1/5 | 0,4534 | 0,975 | 1:36 |
| 2/5 | 0,09553 | 0,979 | 1:31 |
| 3/5 | 0,0762 | 0,984 | 1:29 |
| 4/5 | 0,05444 | 0,988 | 1:29 |
| 5/5 | 0,03188 | 0,988 | 1:30 |

- `top1_acc` adalah persentase gambar yang tebakan teratasnya benar. `top5_acc` adalah persentase yang jawaban benarnya masuk 5 tebakan teratas. Karena kelasnya hanya 4, jawaban benar pasti masuk 5 teratas, makanya `top5_acc` selalu **1** dan tidak informatif di kasus ini. Yang dilihat adalah `top1_acc`.
- Loss turun terus dari 0,45 menjadi 0,03, dan akurasi validasi naik dari 97,5% menjadi 98,8%. Kurvanya mulus tanpa lonjakan, tidak seperti pada CNN buatan sendiri.
- **Perbandingan dengan Bagian A:** setelah epoch pertama saja YOLO sudah 97,5%, sedangkan CNN dari nol setelah 9 epoch baru sekitar 88,5%. Itulah kekuatan transfer learning: model sudah punya fitur visual yang bagus sejak awal.
- Setiap epoch memproses 369/369 batch dalam sekitar 1,5 menit. Total 5 epoch selesai dalam 0,173 jam, sekitar 10,4 menit.

**3. Penyimpanan.** `best.pt` (model dengan hasil validasi terbaik) dan `last.pt` (model epoch terakhir) disimpan masing-masing 112,5 MB di `runs/classify/train/weights/`. Tulisan *Optimizer stripped* artinya data optimizer dibuang dari file agar ukuran lebih kecil, karena tidak diperlukan saat inferensi. Setelah training, YOLO otomatis memvalidasi `best.pt` dan hasilnya `top1_acc = 0,988`.

In [ ]:
# prompt: show image from runs/classify/train/result.png

from IPython.display import Image
Image('/content/runs/classify/train/results.png', width=750)


**Hasil dan analisis (grafik `results.png`).** Gambar ini dibuat otomatis oleh YOLO, isinya empat grafik dengan sumbu x berupa epoch 1 sampai 5. Garis putus-putus oranye (*smooth*) adalah versi yang dihaluskan dari garis biru.

| Grafik | Yang terlihat | Artinya |
|---|---|---|
| `train/loss` | turun tajam dari sekitar 0,45 ke sekitar 0,03 | model makin sedikit salah di data latih |
| `val/loss` | turun terus dari sekitar 0,775 ke sekitar 0,756 | loss di data validasi juga membaik, tidak ada tanda overfitting |
| `metrics/accuracy_top1` | naik dari sekitar 0,975 ke sekitar 0,988 | akurasi bertambah tiap epoch |
| `metrics/accuracy_top5` | datar di 1,00 | selalu 100% karena hanya ada 4 kelas |

Skala `val/loss` berbeda dari `train/loss` sehingga keduanya tidak bisa dibandingkan angkanya langsung. Yang perlu diperhatikan adalah **arah garisnya**, yaitu sama-sama turun. Tidak ada garis validasi yang berbalik naik sementara training turun, jadi tidak ada tanda overfitting. Akurasi juga belum sepenuhnya mendatar di epoch 5, artinya kalau epoch ditambah mungkin masih ada sedikit perbaikan.

## B.4 Validasi Model Custom

**Tujuan.** Mengukur performa model terbaik (`best.pt`) secara terpisah, memakai data validasi.

**Penjelasan kode.**

`!yolo task=classify mode=val model=/content/runs/classify/train/weights/best.pt data={dataset.location}`
- `mode=val`: mode validasi.
- `model=.../best.pt`: memakai bobot terbaik hasil training tadi.
- `data={dataset.location}`: dataset yang sama. Secara bawaan YOLO memakai split `val` untuk validasi.

In [ ]:
!yolo task=classify mode=val model=/content/runs/classify/train/weights/best.pt data={dataset.location}

**Hasil dan analisis.**
- `YOLOv8x-cls summary (fused): 133 layers, 56.128.324 parameters`: saat validasi, model berukuran 133 layer. Di training tadi terlihat 183 layer dan 56.146.964 parameter. Bedanya karena layer konvolusi dan batch norm digabung (*fused*) agar inferensi lebih ringan.
- Ditemukan `11793` gambar train, `1123` gambar val, dan `562` gambar test, masing-masing dalam 4 kelas, sama dengan jumlah di A.3.
- `top1_acc = 0,988` dan `top5_acc = 1`. Artinya sekitar 98,8% dari 1.123 gambar validasi (kira-kira 1.109 gambar) ditebak benar dalam satu kali tebakan, dan hanya sekitar 14 gambar yang salah. Angka ini sama dengan akurasi epoch terakhir, jadi konsisten.
- Kecepatan: 0,1 ms preprocess dan 4,9 ms inferensi per gambar, sangat cepat.
- Hasil disimpan di `runs/classify/val`, termasuk gambar confusion matrix yang ditampilkan di sel berikut.

In [ ]:
Image('/content/runs/classify/val/confusion_matrix_normalized.png', width=750)

**Membaca confusion matrix ternormalisasi (YOLO).**

Perhatikan bahwa orientasinya **berbeda** dari confusion matrix di Bagian A. Di gambar YOLO ini, **sumbu x (kolom) adalah label asli (True)** dan **sumbu y (baris) adalah label prediksi (Predicted)**. *Normalized* artinya angkanya sudah dijadikan proporsi, jadi setiap kolom jumlahnya 1 (100% dari gambar kelas asli tersebut). Baris dan kolom `background` kosong karena tugas klasifikasi tidak punya kelas latar.

| Kelas asli (kolom) | Diprediksi benar | Salah ke kelas lain |
|---|---|---|
| overripe | 0,99 | 0,01 ke rotten |
| ripe | 0,99 | 0,01 ke rotten |
| rotten | 0,98 | 0,01 ke overripe dan 0,01 ke unripe |
| unripe | 0,99 | 0,01 ke rotten |

Diagonal berwarna biru tua pekat (0,98 sampai 0,99), sedangkan sel lainnya nyaris putih. Artinya model hampir tidak pernah salah, dan kesalahan kecil yang ada paling banyak bermuara ke kelas `rotten`. Ini masuk akal karena secara visual pisang yang terlalu matang, sudah ada bagian yang menghitam, atau hijau tetapi ujungnya busuk memang mirip dengan pisang busuk.

## B.5 Inferensi dengan Model Custom pada Data Test

**Tujuan.** Menguji `best.pt` pada **data test** (562 gambar). Data ini tidak dipakai untuk training dan tidak dipakai memilih model terbaik, jadi hasilnya paling jujur menggambarkan performa di data baru.

**Penjelasan kode.**

Perintahnya sama untuk empat sel: `!yolo task=classify mode=predict model=.../best.pt conf=0.25 source={dataset.location}/test/<kelas>`.
- Sumber datanya satu folder penuh (bukan satu gambar), jadi YOLO memprediksi semua gambar di folder itu.
- Dibuat empat sel, satu untuk tiap folder (`overripe`, `ripe`, `rotten`, `unripe`). Karena nama folder adalah label aslinya, kita bisa langsung mengecek apakah tebakan YOLO cocok.
- Setiap kali predict dijalankan, YOLO membuat folder hasil baru dengan nomor naik. Predict pertama (anjing di B.2) tersimpan di `predict`, lalu empat sel ini berturut-turut menjadi `predict2` (overripe), `predict3` (ripe), `predict4` (rotten), dan `predict5` (unripe). Nomor ini dipakai lagi di sel visualisasi setelahnya.

### Prediksi folder `test/overripe`

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/overripe

### Prediksi folder `test/ripe`

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/ripe


### Prediksi folder `test/rotten`

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/rotten

### Prediksi folder `test/unripe`

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/unripe

**Hasil dan analisis.**

Setiap baris log berisi nama file, ukuran input (224x224), lalu empat probabilitas diurutkan dari yang tertinggi, dan waktu proses sekitar 14 ms per gambar. Contohnya `overripe 1.00, ripe 0.00, rotten 0.00, unripe 0.00` artinya model sangat yakin gambar itu overripe.

Log tiap folder panjang, jadi saya hitung ringkasannya dari kelas dengan probabilitas tertinggi pada setiap baris:

| Folder (kelas asli) | Jumlah gambar | Ditebak benar | Salah | Rincian salah |
|---|---|---|---|---|
| overripe | 113 | 113 | 0 | tidak ada |
| ripe | 154 | 154 | 0 | tidak ada |
| rotten | 185 | 182 | 3 | 2 ditebak unripe, 1 ditebak ripe |
| unripe | 110 | 110 | 0 | tidak ada |
| **Total** | **562** | **559** | **3** | |

Akurasi di data test = 559 / 562 = **99,47%** (sekitar 99,5%). Jumlah gambar 113 + 154 + 185 + 110 = 562 cocok dengan jumlah data test yang dilaporkan YOLO di B.4.

Hal yang bisa disimpulkan:
1. Hasil di data test (sekitar 99,5%) sejalan dengan akurasi validasi (98,8%). Selisih kecilnya wajar karena isi gambarnya berbeda. Tidak ada penurunan besar, jadi model **tidak overfitting** dan mampu menggeneralisasi ke data yang belum pernah dilihat.
2. Hanya kelas `rotten` yang punya salah tebak, itu pun hanya 3 dari 185. Pola ini searah dengan confusion matrix di B.4, di mana `rotten` adalah kelas yang paling sering terlibat kesalahan.
3. Model hampir selalu sangat yakin (probabilitas 1,00 untuk kelas yang dipilih), yang menandakan fitur tiap kelas dipisahkan dengan jelas.

### Menampilkan Contoh Hasil Prediksi

**Tujuan.** Melihat langsung beberapa gambar hasil prediksi, supaya hasil angka di atas bisa dicek secara visual.

**Penjelasan kode (empat sel di bawah, strukturnya sama).**
- `import glob`: modul untuk mencari file berdasarkan pola nama.
- `from IPython.display import Image, display`: alat untuk menampilkan gambar di notebook.
- `glob.glob('/content/runs/classify/predict2/*.jpg')`: mengumpulkan semua file `.jpg` di folder hasil prediksi (`predict2` untuk overripe, `predict3` ripe, `predict4` rotten, `predict5` unripe). Tanda `*` artinya nama file apa saja.
- `[:3]`: mengambil 3 file pertama saja supaya tidak menampilkan ratusan gambar. Urutan hasil `glob` tidak disortir, jadi "3 pertama" ini sifatnya acak.
- `for ... in ...:` mengulang untuk tiap file, lalu `display(Image(filename=image_path, width=600))` menampilkan gambarnya selebar 600 piksel.

**Contoh hasil kelas overripe (`predict2`)**

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict2/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))



**Contoh hasil kelas ripe (`predict3`)**

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict3/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))

**Contoh hasil kelas rotten (`predict4`)**

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict4/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))

**Contoh hasil kelas unripe (`predict5`)**

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict5/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))

**Hasil dan analisis.** Setiap gambar punya tulisan kecil di pojok kiri atas berisi keempat kelas beserta probabilitasnya, dengan kelas tertinggi di baris paling atas. Dari 12 gambar contoh (3 per kelas), semuanya ditebak benar dengan probabilitas 1,00:

| Kelas | Yang terlihat di gambar | Hasil |
|---|---|---|
| overripe | pisang kuning dengan banyak bercak cokelat sampai kehitaman | overripe 1,00 |
| ripe | pisang kuning (ada yang kekuningan-hijau, ada yang oranye) dengan sedikit bintik kecil | ripe 1,00 |
| rotten | bagian yang menghitam atau membusuk, termasuk pisang hijau yang ujungnya sudah hitam | rotten 1,00 |
| unripe | pisang hijau mulus tanpa bercak | unripe 1,00 |

Beberapa gambar punya sudut hitam atau tampak miring. Itu bekas augmentasi (rotasi) yang sudah dilakukan Roboflow pada dataset, bukan kesalahan model.

Catatan: contoh `rotten` yang berupa pisang hijau dengan ujung hitam menarik karena warnanya sebagian besar hijau seperti `unripe`, tetapi model tetap membaca bagian busuknya. Ini menunjukkan model tidak sekadar mengandalkan warna dominan. Walaupun begitu, 3 gambar per kelas hanya contoh. Ukuran yang lebih bisa diandalkan adalah akurasi pada 562 gambar test (99,5%) di atas.

---
# Bagian C. Perbandingan CNN Buatan Sendiri dan YOLOv8 (Transfer Learning)

| Aspek | CNN buatan sendiri (Bagian A) | YOLOv8x-cls (Bagian B) |
|---|---|---|
| Framework | TensorFlow / Keras | PyTorch (Ultralytics) |
| Cara melatih | dari nol, bobot acak | transfer learning dari model pretrained, `freeze=5` |
| Jumlah parameter | 11.169.476 (sekitar 11 juta) | sekitar 56 juta |
| Jumlah epoch | 9 | 5 |
| Waktu training | sekitar 14 menit (sekitar 2,5 menit per epoch) | sekitar 10 menit (sekitar 1,5 menit per epoch) |
| Optimizer | Adam (diatur manual) | AdamW (dipilih otomatis, lr sekitar 0,0007) |
| Augmentasi | diatur manual lewat `ImageDataGenerator` | bawaan Ultralytics |
| Akurasi validasi | sekitar 88,5% | sekitar 98,8% |
| Akurasi data test | tidak diuji | sekitar 99,5% (559 dari 562) |
| Stabilitas training | naik-turun | mulus |
| Evaluasi per kelas | classification report (hasilnya keliru karena urutan data teracak, lihat A.6) | confusion matrix ternormalisasi, diagonal 0,98 sampai 0,99 |

**Kenapa YOLOv8 jauh lebih baik?**
1. **Sudah pretrained.** Fitur dasar seperti tepi, tekstur, dan pola sudah dipelajari dari dataset yang sangat besar, jadi tidak mulai dari nol (keuntungan transfer learning di slide). Akurasi 97,5% sudah dicapai di epoch pertama.
2. **Model lebih dalam dan besar.** Sekitar 56 juta parameter dengan 183 layer, versus 3 blok konvolusi di CNN buatan sendiri. Kapasitas lebih besar membuatnya bisa mengenali perbedaan halus, misalnya antara `ripe` dan `overripe`.
3. **Freezing menjaga stabilitas.** Layer awal tidak diubah, sehingga training tidak mudah goyah dan lebih cepat (tidak perlu menghitung gradien untuk layer yang dibekukan).
4. **Pengaturan training yang sudah matang.** Optimizer, learning rate, dan AMP dipilih otomatis oleh Ultralytics.

**Catatan jujur.** Perbandingannya tidak sepenuhnya setara. CNN buatan sendiri punya kendala teknis (masalah batch di A.5) dan hanya dilatih 9 epoch dengan arsitektur sederhana. Walaupun begitu, arah kesimpulannya jelas: untuk dataset seperti ini, **transfer learning memberi hasil yang jauh lebih baik dengan waktu yang lebih singkat**.

**Kelebihan CNN buatan sendiri.** Ukuran modelnya lebih kecil dan arsitekturnya bisa dipahami serta diubah-ubah sepenuhnya, sehingga bagus sebagai bahan belajar. Kalau sumber daya terbatas (misalnya harus berjalan di perangkat kecil), model sederhana seperti ini masih bisa menjadi pilihan.

---
# Bagian D. Kesimpulan

1. **Alur kerja klasifikasi gambar** pada praktikum ini adalah: cek GPU, unduh dataset, preprocessing dan augmentasi, bangun atau pilih model, training, evaluasi, lalu inferensi. Alur ini sama untuk CNN dan YOLO. Bedanya ada di cara model disiapkan.
2. **CNN buatan sendiri** (Conv2D, MaxPooling, Flatten, Dense, Dropout, Softmax) berhasil belajar mengenali 4 tingkat kematangan pisang dengan akurasi validasi sekitar **88,5%**, walaupun dilatih dari nol hanya 9 epoch. Grafik training-nya tidak mulus karena `steps_per_epoch` tidak pas dengan jumlah batch.
3. **Classification report dan confusion matrix di Bagian A** menunjukkan akurasi 0,30 yang tidak sesuai dengan `evaluate` (0,885). Penyebabnya diduga urutan data validasi yang teracak (`shuffle=True`), bukan kualitas model. Perbaikannya cukup dengan `shuffle=False`.
4. **YOLOv8x-cls dengan transfer learning** (`freeze=5`, 5 epoch) mencapai akurasi validasi **98,8%** dan akurasi test sekitar **99,5%** (559 dari 562 gambar benar), dengan training yang stabil dan lebih singkat.
5. **Transfer learning terbukti efektif**: memakai pengetahuan dari model pretrained menghemat waktu, mengurangi kebutuhan epoch, dan meningkatkan akurasi, sesuai keuntungan yang dijelaskan di slide teori.
6. Hampir semua kesalahan YOLO terjadi pada kelas `rotten`, karena pisang yang terlalu matang, sudah menghitam sebagian, atau punya bagian busuk bisa tampak mirip secara visual.

---
# Bagian E. Catatan Temuan dan Saran Perbaikan

Saat membaca output, saya menemukan dua hal kecil di Bagian A yang menurut saya perlu dicatat. Kode asli dosen sengaja tidak saya ubah di notebook ini, jadi perbaikannya saya tuliskan di sini sebagai saran (tidak dijalankan).

## 1. Urutan data validasi teracak (mempengaruhi A.6)
`valid_generator` dibuat dengan `shuffle=True` (bawaan), sehingga `model.predict(valid_generator)` dan `valid_generator.classes` tidak sinkron. Perbaikannya:

```python
valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False          # urutan data dijaga tetap supaya cocok dengan label asli
)
```
Setelah itu `predict`, `classification_report`, dan `confusion_matrix` dijalankan ulang. Hasil yang diharapkan adalah akurasi di report mendekati 0,88 dan angka confusion matrix terkonsentrasi di diagonal.

## 2. `steps_per_epoch` tidak pas dengan jumlah batch (mempengaruhi A.5 dan A.7)
Jumlah batch train adalah 369 (11.793 / 32 dibulatkan ke atas) dan validasi 36 (1.123 / 32 dibulatkan ke atas), tetapi kode memakai pembagian bulat ke bawah (`//`) yaitu 368 dan 35. Cara yang lebih aman adalah tidak mengisi `steps_per_epoch` dan `validation_steps` sama sekali (Keras akan memakai jumlah batch lengkap dari generator), atau memakai pembulatan ke atas:

```python
import math
history = model.fit(
    train_generator,
    steps_per_epoch=math.ceil(train_generator.samples / train_generator.batch_size),
    validation_data=valid_generator,
    validation_steps=math.ceil(valid_generator.samples / valid_generator.batch_size),
    epochs=9
)
```
Dengan begitu tidak ada lagi epoch genap yang terpotong dan grafik akurasi serta loss akan lebih mulus dan lebih mudah dibaca.

## 3. Saran umum untuk pengembangan
- Menambah jumlah epoch dan memakai `EarlyStopping` (seperti pada praktikum MLP) agar training berhenti saat loss validasi tidak membaik.
- Mencoba transfer learning di Keras (misalnya `ResNet50` atau `VGG16`, dua arsitektur yang disebut di slide) sebagai pembanding langsung terhadap YOLOv8.
- Menguji CNN buatan sendiri pada data test (562 gambar) supaya perbandingannya setara dengan YOLO.